# Retrieval-Augmented Generation (RAG)
The objective of this notebook is to build a complete Retrieval-Augmented Generation (RAG) pipeline for the Smart MCQ Solver Challenge.

The pipeline includes:

- Creating a Knowledge Base
- Dense Embedding Generation using Sentence Transformers
- FAISS Vector Search
- Cross-Encoder Reranking
- Zero-shot Classification
- Retrieval-Augmented Question Answering
- Performance Evaluation using MAP@3

#  1. Import Required Libraries

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

# Data Manipulation
import pandas as pd
import numpy as np

# Vector Database
import faiss

# Embedding Models
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

# Hugging Face
from transformers import (
    AutoTokenizer,
    pipeline
)

# Classical NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

e:\Projects\GenAi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Load Dataset

In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

train = pd.read_csv("../data/train.csv")

print("Dataset Shape :", train.shape)

train.head()

Dataset Shape : (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# 3. Create Knowledge Base

## Objective

Create a Knowledge Base (KB) using only the correct answers from the
training dataset.

Each document in the KB corresponds to the correct option
(A, B, C, D or E) of a training example.

This KB will later be indexed using FAISS for efficient retrieval.

In [3]:
# ==========================================================
# Create Knowledge Base
# ==========================================================

print("Creating Knowledge Base...")

kb = []

for _, row in train.iterrows():

    correct_letter = row["answer"]

    correct_document = str(
        row[correct_letter]
    )

    kb.append(correct_document)

print(f"Total Documents : {len(kb)}")
print("Knowledge Base Created Successfully ✅")

Creating Knowledge Base...
Total Documents : 2000
Knowledge Base Created Successfully ✅


# 4. Create Dense Embeddings & FAISS Index

## Objective

Convert every knowledge base document into dense vector embeddings
using Sentence Transformers.

These embeddings are then stored inside a FAISS vector index to
enable fast semantic similarity search.

In [4]:
# ==========================================================
# Load Sentence Transformer
# ==========================================================

print("Loading Embedding Model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# ==========================================================
# Generate Dense Embeddings
# ==========================================================

kb_embeddings = embedding_model.encode(
    kb,
    show_progress_bar=False
)

# ==========================================================
# Create FAISS Index
# ==========================================================

index = faiss.IndexFlatL2(
    kb_embeddings.shape[1]
)

index.add(kb_embeddings)

print("FAISS Index Created Successfully ✅")

Loading Embedding Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3174.82it/s]


FAISS Index Created Successfully ✅


# 5. Initialize Zero-shot Classifier

## Objective

Load the Facebook BART Large MNLI model.

This model will be used for zero-shot multiple-choice classification
throughout this milestone.

In [5]:
# ==========================================================
# Initialize Zero-shot Classifier
# ==========================================================

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Zero-shot Classifier Loaded Successfully ✅")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2425.30it/s]


Zero-shot Classifier Loaded Successfully ✅


# 6. Prepare Sample (Row 150)

## Objective

Extract the prompt, answer options, and ground-truth answer
from row index 150.

This sample will be used in Questions 1, 2, 5, and 6.

In [6]:
# ==========================================================
# Prepare Sample
# ==========================================================

row_150 = train.iloc[150]

prompt_150 = str(
    row_150["prompt"]
)

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

true_document = str(
    row_150[row_150["answer"]]
)

print("Sample Prepared Successfully ✅")

Sample Prepared Successfully ✅


# Question 1 : Zero-shot Classification

## Objective

Use the **facebook/bart-large-mnli** zero-shot classifier to predict the probability assigned to the correct answer for row index **150**.

This experiment establishes the baseline prediction confidence **before applying Retrieval-Augmented Generation (RAG)**.

In [ ]:
# ==========================================================
# Question 1
# Baseline Zero-shot Classification
# ==========================================================

# Run zero-shot classification
result = zs(
    prompt_150,
    candidate_labels=labels_150
)

# Ground truth answer
true_answer = row_150["answer"]
true_text = str(row_150[true_answer])

# Extract probability of the correct option
ground_truth_score = result["scores"][
    result["labels"].index(true_text)
]

print("Ground Truth Option :", true_answer)
print("Probability :", round(ground_truth_score, 3))

Ground Truth Option : C
Probability : 0.384


### Conclusion

The baseline confidence score for the correct answer is **0.384**.

This value serves as the reference point for later RAG experiments. In subsequent questions, retrieved contextual information will be injected into the prompt to determine whether retrieval augmentation improves the model's confidence and prediction quality.

# Question 2 : Dense Retrieval using FAISS

## Objective

Embed the prompt from **row index 150** using the **all-MiniLM-L6-v2** sentence transformer.

Use the generated embedding to query the **FAISS vector database** and retrieve the **Top-10 most similar documents**.

Finally, determine the exact rank (1–10) at which the **ground-truth document** (originally stored at KB index **150**) appears.

In [8]:
# ==========================================================
# Question 2
# Dense Retrieval using FAISS
# ==========================================================

# Encode the prompt into a dense embedding
query_embedding = embedding_model.encode(
    [prompt_150],
    show_progress_bar=False
)

# Retrieve Top-10 nearest documents
distances, retrieved_indices = index.search(
    query_embedding,
    k=10
)

retrieved_indices = retrieved_indices[0]

print("Retrieved KB Indices:")
print(retrieved_indices)

# Find rank of the true document (KB index 150)
true_rank = None

for rank, idx in enumerate(retrieved_indices, start=1):
    if idx == 150:
        true_rank = rank
        break

print(f"\nGround Truth Document Rank : {true_rank}")

Retrieved KB Indices:
[ 663 1701 1269 1532  576  847 1693 1906  168  150]

Ground Truth Document Rank : 10


### Conclusion

The ground-truth document was retrieved at **Rank 10** by the FAISS vector database.

Although the correct document exists within the retrieved Top-10 results, its low ranking highlights the limitations of dense retrieval using only embedding similarity. This motivates the use of a **Cross-Encoder** in the next stage, which jointly analyzes the query and retrieved documents to improve ranking accuracy.

# Question 3 : Cross-Encoder Re-ranking

## Objective

Take the **Top-10 documents** retrieved by FAISS in Question 2 and rerank them using the **cross-encoder/ms-marco-MiniLM-L-6-v2** model.

Finally, determine the new rank (1–10) of the **ground-truth document** after reranking.

In [9]:
# ==========================================================
# Question 3
# Cross-Encoder Re-ranking
# ==========================================================

# Load Cross Encoder
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# Retrieve the Top-10 documents from FAISS
docs_10 = [kb[i] for i in retrieved_indices]

# Create (query, document) pairs
pairs = [
    [prompt_150, doc]
    for doc in docs_10
]

# Compute Cross-Encoder relevance scores
ce_scores = cross_encoder.predict(pairs)

# Sort documents according to score
reranked = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Re-ranked Documents:\n")

for rank, (idx, score) in enumerate(reranked, start=1):
    print(
        f"Rank {rank:2d} | KB Index: {idx:4d} | Score: {score:.4f}"
    )

# Find the new rank of the ground-truth document
true_rank = None

for rank, (idx, _) in enumerate(reranked, start=1):
    if idx == 150:
        true_rank = rank
        break

print(f"\nGround Truth Rank after Re-ranking : {true_rank}")

e:\Projects\GenAi\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ps928\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2643.17it/s]


Re-ranked Documents:

Rank  1 | KB Index:  150 | Score: 4.7585
Rank  2 | KB Index:  847 | Score: 4.7526
Rank  3 | KB Index: 1693 | Score: 4.7526
Rank  4 | KB Index: 1906 | Score: 4.7526
Rank  5 | KB Index: 1269 | Score: 4.7375
Rank  6 | KB Index: 1532 | Score: 4.7375
Rank  7 | KB Index:  168 | Score: 4.7072
Rank  8 | KB Index:  576 | Score: 4.6870
Rank  9 | KB Index:  663 | Score: 4.6602
Rank 10 | KB Index: 1701 | Score: 4.6602

Ground Truth Rank after Re-ranking : 1


### Conclusion

The Cross-Encoder successfully improved the ranking of the correct document from **Rank 10** to **Rank 1**.

This experiment validates the effectiveness of the two-stage Retrieval-Augmented Generation (RAG) pipeline, where:

- **FAISS** provides fast semantic retrieval.
- **Cross-Encoder** refines the retrieved results by performing deeper semantic comparison.

Together, they produce significantly more accurate retrieval than using FAISS alone.

# Question 4 : Tokenization of Retrieved Context

## Objective

Retrieve the **Top-5** most relevant documents for the prompt at **row index 42** using FAISS.

Concatenate these retrieved documents into a single context and combine it with the original prompt.

Finally, tokenize the complete RAG input using the **bert-base-uncased** tokenizer and determine the total number of generated tokens.

In [10]:
# ==========================================================
# Question 4
# Token Count for Retrieved Context
# ==========================================================

# Load row 42
row_42 = train.iloc[42]

prompt_42 = str(row_42["prompt"])

# Generate embedding for the prompt
query_embedding = embedding_model.encode(
    [prompt_42],
    show_progress_bar=False
)

# Retrieve Top-5 documents
_, retrieved_indices = index.search(
    query_embedding,
    k=5
)

retrieved_indices = retrieved_indices[0]

# Collect retrieved documents
docs_5 = [
    kb[idx]
    for idx in retrieved_indices
]

# Concatenate documents
context = " ".join(docs_5)

# Build RAG input
rag_text = f"Context: {context} Question: {prompt_42}"

# Load BERT tokenizer
bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

# Tokenize WITHOUT truncation
tokens = bert_tokenizer(
    rag_text,
    truncation=False
)

# Count total tokens
total_tokens = len(tokens["input_ids"])

print("Retrieved Indices :", retrieved_indices)
print("Total Tokens :", total_tokens)

Retrieved Indices : [  42  538  822 1247 1574]
Total Tokens : 216


### Conclusion

The complete Retrieval-Augmented Generation (RAG) input produced **216 tokens** after BERT tokenization.

This experiment demonstrates that adding retrieved contextual information increases the input sequence length. Therefore, selecting only the most relevant documents is important to balance contextual richness with computational efficiency.

# Question 5 : Retrieval-Augmented Zero-shot Classification

## Objective

Retrieve the **ground-truth document** for **row index 150** from the Knowledge Base and use it as external context.

Construct a Retrieval-Augmented Generation (RAG) prompt and perform zero-shot classification using **facebook/bart-large-mnli**.

Finally, compare the confidence of the correct answer with the baseline obtained in Question 1.

In [11]:
# ==========================================================
# Question 5
# Zero-shot Classification with Correct Retrieved Context
# ==========================================================

# Retrieve the true document
true_document = kb[150]

# Build the RAG prompt
rag_prompt = (
    f"Context: {true_document} "
    f"Question: {prompt_150}"
)

# Run zero-shot classification
result = zs(
    rag_prompt,
    candidate_labels=labels_150
)

# Ground-truth option
true_answer = row_150["answer"]
true_text = str(row_150[true_answer])

# Probability assigned to the correct answer
ground_truth_score = result["scores"][
    result["labels"].index(true_text)
]

print("Ground Truth Option :", true_answer)
print("Probability :", round(ground_truth_score, 3))

Ground Truth Option : C
Probability : 0.989


### Conclusion

Providing the correct contextual document dramatically increased the model's confidence from **0.384** to **0.989**.

This experiment clearly demonstrates the primary objective of Retrieval-Augmented Generation (RAG): supplying relevant external knowledge enables the language model to make more accurate and confident predictions without modifying the underlying model parameters.

# Question 6 : Adversarial Retrieval-Augmented Generation (RAG)

## Objective

Investigate the effect of providing an **incorrect retrieved document** as context.

Instead of using the correct document, we intentionally inject an unrelated document from the knowledge base and observe how the confidence of the zero-shot classifier changes.

In [12]:
# ==========================================================
# Question 6
# Adversarial Retrieval-Augmented Generation
# ==========================================================

# Intentionally use an unrelated document
wrong_document = kb[999]

# Construct the adversarial RAG prompt
adversarial_prompt = (
    f"Context: {wrong_document} "
    f"Question: {prompt_150}"
)

# Perform zero-shot classification
result = zs(
    adversarial_prompt,
    candidate_labels=labels_150
)

# Extract the probability of the correct answer
true_answer = row_150["answer"]
true_text = str(row_150[true_answer])

ground_truth_score = result["scores"][
    result["labels"].index(true_text)
]

print("Ground Truth Option :", true_answer)
print("Probability :", round(ground_truth_score, 3))

Ground Truth Option : C
Probability : 0.529


### Conclusion

Injecting an unrelated document reduced the confidence of the correct answer from **0.989** to **0.529**.

This experiment highlights the importance of retrieval quality in Retrieval-Augmented Generation (RAG). While external knowledge can significantly improve model confidence when relevant, irrelevant or misleading context can negatively influence predictions. Therefore, accurate retrieval is essential for building reliable RAG systems.

# Question 7 : Retrieval Hit Rate Evaluation

## Objective

Evaluate the retrieval performance of the FAISS vector database.

For the first **100 samples** in the training dataset, retrieve the **Top-5** most similar documents. A retrieval is considered a **Hit** if the exact ground-truth document is present among the retrieved Top-5 documents.

Finally, compute the overall **Hit Rate (%)**.

In [13]:
# ==========================================================
# Question 7
# Evaluate Retrieval Hit Rate
# ==========================================================

hits = 0
total = 100

for i in range(total):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    # Ground-truth document
    true_doc = str(row[row["answer"]])

    # Encode prompt
    query_embedding = embedding_model.encode(
        [prompt],
        show_progress_bar=False
    )

    # Retrieve Top-5 documents
    _, retrieved_indices = index.search(
        query_embedding,
        k=5
    )

    retrieved_docs = [
        kb[idx]
        for idx in retrieved_indices[0]
    ]

    # Check if ground-truth document is present
    if true_doc in retrieved_docs:
        hits += 1

# Calculate Hit Rate
hit_rate = (hits / total) * 100

print(f"Hits      : {hits}")
print(f"Hit Rate  : {hit_rate:.1f}%")

Hits      : 73
Hit Rate  : 73.0%


### Conclusion

The FAISS retriever achieved a **Hit Rate of 73.0%** on the first 100 samples.

This result shows that the retrieval component is reasonably effective but not perfect. Since nearly one-quarter of the queries fail to retrieve the correct document within the Top-5 results, additional techniques such as **Cross-Encoder reranking** are essential for improving the overall performance of a Retrieval-Augmented Generation (RAG) pipeline.

# Question 8 : Complete Retrieval-Augmented Generation (RAG) Pipeline

## Objective

Build a complete Retrieval-Augmented Generation (RAG) pipeline for the first **20 questions** of the training dataset.

The pipeline consists of:

1. Dense Retrieval using Sentence Transformers and FAISS
2. Cross-Encoder Re-ranking
3. Retrieval-Augmented Prompt Construction
4. Zero-shot Classification
5. MAP@3 Evaluation

Finally, compute the average MAP@3 score across all 20 samples.

In [14]:
# ==========================================================
# Question 8
# Complete Retrieval-Augmented Generation Pipeline
# ==========================================================

# MAP@3 function
def map_at_3(actual, prediction):

    if actual in prediction:
        rank = prediction.index(actual) + 1
        return 1 / rank

    return 0


scores = []

# Process first 20 rows
for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])
    actual = row["answer"]

    options = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"])
    }

    # ------------------------------------------------------
    # Step 1 : Dense Retrieval
    # ------------------------------------------------------

    query_embedding = embedding_model.encode(
        [prompt],
        show_progress_bar=False
    )

    _, retrieved_indices = index.search(
        query_embedding,
        k=5
    )

    retrieved_indices = retrieved_indices[0]

    docs_5 = [
        kb[idx]
        for idx in retrieved_indices
    ]

    # ------------------------------------------------------
    # Step 2 : Cross-Encoder Re-ranking
    # ------------------------------------------------------

    pairs = [
        [prompt, doc]
        for doc in docs_5
    ]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs_5[
        np.argmax(ce_scores)
    ]

    # ------------------------------------------------------
    # Step 3 : Build RAG Prompt
    # ------------------------------------------------------

    rag_prompt = (
        f"Context: {best_doc} "
        f"Question: {prompt}"
    )

    # ------------------------------------------------------
    # Step 4 : Zero-shot Classification
    # ------------------------------------------------------

    candidate_labels = list(options.values())

    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels
    )

    # ------------------------------------------------------
    # Step 5 : Convert Text Predictions to Letters
    # ------------------------------------------------------

    predicted_letters = []

    for label in result["labels"]:

        for letter, text in options.items():

            if label == text:
                predicted_letters.append(letter)

    top3 = predicted_letters[:3]

    score = map_at_3(
        actual,
        top3
    )

    scores.append(score)

# ==========================================================
# Final MAP@3
# ==========================================================

final_map3 = np.mean(scores)

print(f"Final MAP@3 : {final_map3:.3f}")

Final MAP@3 : 0.975
